# Visualización I — Sesión 22
## Seaborn II: facetas, correlaciones y regresiones visuales
### Docente: Adriana Pineda

**Propósito de la sesión.** Usar Seaborn para pasar de gráficos descriptivos aislados a un análisis visual multivariado, trabajando con **facetas, `pairplot`, `heatmap` y `lmplot`**. A lo largo de la sesión se discutirá cuándo una visualización aporta claridad y cuándo introduce ruido interpretativo.

### Pregunta crítica de la sesión

> **¿Qué patrones territoriales y relaciones entre volumen, perfil de los casos y prioridad de las emergencias pueden identificarse, y cuáles de esos patrones son reales frente a cuáles podrían ser artefactos de problemas de calidad de los datos?**

Esta pregunta guiará toda la clase. No comenzaremos limpiando la base completa. Primero veremos qué nos dicen —y qué nos ocultan— los datos tal como llegan. Después limpiaremos **solo las variables necesarias para responder cada pregunta analítica**.

### Conexión con la Sesión 10

Antes de construir cualquier gráfico preguntaremos:

1. ¿Qué pregunta analítica queremos responder?
2. ¿Qué variables necesitamos?
3. ¿Son numéricas, categóricas o temporales?
4. ¿Necesitamos comparar grupos, estudiar asociaciones o ambas cosas?
5. ¿La visualización elegida facilita la lectura o añade complejidad innecesaria?

**Idea clave:** Seaborn no decide qué gráfico debemos usar. Facilita la construcción; la decisión sigue siendo analítica y comunicativa.

## 1. Preparación del entorno

Trabajaremos con:

- `llamadas123_2025.xlsx`: registros de diciembre de 2025.
- `llamadas123(3).xlsx`: registros de 2026.
- `localidades(2).csv`: información territorial de las localidades.

En Google Colab, cargue los tres archivos cuando se ejecute la siguiente celda.

In [ ]:
from google.colab import files
archivos = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

archivo_2025 = "llamadas123_2025.xlsx"
archivo_2026 = "llamadas123.xlsx"
archivo_localidades = "localidades.csv"

df_2025 = pd.read_excel(archivo_2025)
df_2026 = pd.read_excel(archivo_2026, sheet_name="llamadas123")
localidades = pd.read_csv(archivo_localidades, skipinitialspace=True)

print("2025:", df_2025.shape)
print("2026:", df_2026.shape)
print("Localidades:", localidades.shape)

## 2. Antes de limpiar: conocer lo que realmente recibimos

Una práctica frecuente es limpiar automáticamente antes de mirar. En visualización exploratoria esto puede ocultar una parte importante del problema.

Primero observemos estructura, tipos de datos, valores faltantes y categorías. **Todavía no corregiremos nada.**

In [ ]:
display(df_2026.head())
df_2026.info()

In [ ]:
faltantes = (
    df_2026.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("faltantes")
    .to_frame()
)
faltantes["porcentaje"] = 100 * faltantes["faltantes"] / len(df_2026)
display(faltantes.round(2))

print("Filas duplicadas exactas:", df_2026.duplicated().sum())

In [ ]:
#Frecuencias absolutas

for variable in ["LOCALIDAD", "UNIDAD", "GENERO", "PRIORIDAD_FINAL"]:
    print("\n", "="*15, variable, "="*15)
    print(df_2026[variable].value_counts(dropna=False).head(30))

### Discusión guiada

Antes de continuar:

- ¿`ENGATIVA` y `ENGATIVµ` representan dos localidades diferentes?
- ¿`SAN CRISTOBAL` y `SAN CRISTàBAL` deberían aparecer como categorías distintas?
- ¿Qué significa encontrar simultáneamente `Años`, `A¤os`, `Meses`, `Dias` y `Horas`?
- ¿Por qué `PRIORIDAD_FINAL` contiene tanto textos como números?
- Si construimos un gráfico directamente, ¿Seaborn puede saber cuáles categorías representan lo mismo?

**Conclusión esperada:** un gráfico puede ejecutarse correctamente y, aun así, comunicar una estructura de datos incorrecta.

# Parte I. Facetas: comparar sin mezclar

## 3. Pregunta analítica 1

> **¿Cómo se distribuye el volumen de emergencias entre localidades y qué problemas aparecen si usamos directamente las categorías originales?**

Primero construiremos la visualización **sin limpiar**.

In [ ]:
plt.figure(figsize=(11, 8))

orden = df_2026["LOCALIDAD"].value_counts().index

sns.countplot(
    data=df_2026,
    y="LOCALIDAD",
    order=orden
)

plt.title("Emergencias por localidad — datos originales")
plt.xlabel("Número de registros")
plt.ylabel("Localidad")
plt.tight_layout()
plt.show()

### ¿Qué observamos?

El problema no es estético. Una misma localidad puede estar fragmentada en varias barras por diferencias de codificación. Si interpretáramos este gráfico sin revisar los datos, podríamos comparar como diferentes categorías que territorialmente son la misma.

### Decisión de limpieza

Para la dimensión territorial **no corregiremos manualmente cada nombre**. Usaremos `CODIGO_LOCALIDAD` como llave y la tabla oficial de localidades para recuperar un nombre territorial consistente, además de zona y subred de salud.

Esta decisión es más reproducible que reemplazar nombres uno por uno.

In [ ]:
# Limpiar espacios en los nombres de columnas de la dimensión
localidades.columns = localidades.columns.str.strip()

# Homogeneizar la llave como texto
df_2026["CODIGO_LOCALIDAD"] = df_2026["CODIGO_LOCALIDAD"].astype(str).str.strip()
df_2025["CODIGO_LOCALIDAD"] = df_2025["CODIGO_LOCALIDAD"].astype(str).str.strip()
localidades["CODIGO_LOCALIDAD"] = localidades["CODIGO_LOCALIDAD"].astype(str).str.strip()

# Conservar el nombre original para poder comparar
df_2026 = df_2026.rename(columns={"LOCALIDAD": "LOCALIDAD_ORIGINAL"})
df_2025 = df_2025.rename(columns={"LOCALIDAD": "LOCALIDAD_ORIGINAL"})

columnas_localidad = [
    "CODIGO_LOCALIDAD", "LOCALIDAD", "ZONA", "SUBRED_SALUD", "ORDEN_ZONA"
]

df_2026 = df_2026.merge(
    localidades[columnas_localidad],
    on="CODIGO_LOCALIDAD",
    how="left"
)

df_2025 = df_2025.merge(
    localidades[columnas_localidad],
    on="CODIGO_LOCALIDAD",
    how="left"
)

display(
    df_2026[
        ["CODIGO_LOCALIDAD", "LOCALIDAD_ORIGINAL", "LOCALIDAD"]
    ].drop_duplicates().head(30)
)

### Ahora sí: el mismo gráfico con la dimensión territorial limpia

La comparación antes/después permite ver que **limpiar cambia la representación y, por tanto, puede cambiar la interpretación**.

In [ ]:
plt.figure(figsize=(11, 7))

orden = df_2026["LOCALIDAD"].value_counts().index

sns.countplot(
    data=df_2026,
    y="LOCALIDAD",
    order=orden
)

plt.title("Emergencias por localidad — dimensión territorial depurada")
plt.xlabel("Número de registros")
plt.ylabel("Localidad")
plt.tight_layout()
plt.show()

## 4. ¿Qué son las facetas?

Hasta ahora hemos construido cada visualización en un único espacio gráfico. Sin embargo, en ocasiones queremos analizar el mismo fenómeno para diferentes grupos.

Facetar consiste en dividir una visualización en varios paneles utilizando las categorías de una variable. Cada panel conserva la misma estructura gráfica, pero representa un subconjunto diferente de los datos.

Por ejemplo, si queremos analizar la evolución mensual de las emergencias, podemos construir un panel independiente para cada subred de salud. Así podemos comparar si el comportamiento temporal es similar o diferente entre las subredes.

Las facetas son especialmente útiles cuando incluir todos los grupos en un único gráfico produciría superposición o dificultaría la lectura.
Pero más paneles no significan automáticamente una mejor visualización. Si existen demasiadas categorías, el facetado también puede generar ruido visual.

## ¿Por qué usar facetas?

Un único gráfico puede volverse difícil de leer cuando queremos comparar simultáneamente territorio, tiempo y grupos.

Una **faceta** divide la visualización en pequeños paneles que comparten estructura. La ventaja no es simplemente “tener varios gráficos”: permite comparar el mismo fenómeno bajo una condición categórica común.

### Pregunta analítica 2

> **¿La evolución mensual del número de emergencias presenta el mismo patrón en todas las subredes de salud?**

Aquí combinamos:

- variable temporal: mes;
- variable numérica: número de emergencias;
- variable categórica: subred de salud.

Primero debemos incorporar diciembre de 2025 y construir las variables temporales necesarias.

In [ ]:
# Unir diciembre de 2025 con los registros de 2026
df = pd.concat([df_2025, df_2026], ignore_index=True)

df["FECHA"] = pd.to_datetime(
    df["FECHA_INICIO_DESPLAZAMIENTO_MOVIL"],
    errors="coerce"
)

df["PERIODO"] = df["FECHA"].dt.to_period("M").astype(str)

print(df["PERIODO"].value_counts().sort_index())

### Visualización sin depurar el alcance territorial

Antes de filtrar categorías especiales (`SIN_D`, `FUERA_DE_BOGOTA`), construimos las facetas. Observe si todos los paneles representan realmente una subred comparable.

In [ ]:
conteo_sucio = (
    df.groupby(["PERIODO", "SUBRED_SALUD"], dropna=False)
      .size()
      .reset_index(name="EMERGENCIAS")
)

g = sns.relplot(
    data=conteo_sucio,
    x="PERIODO",
    y="EMERGENCIAS",
    col="SUBRED_SALUD",
    col_wrap=3,
    kind="line",
    marker="o",
    height=3.2,
    aspect=1.15
)

g.set_axis_labels("Periodo", "Número de emergencias")
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=45)

g.figure.suptitle(
    "Evolución mensual por subred — antes de depurar categorías",
    y=1.04
)
plt.show()

### Limpieza necesaria para esta pregunta

La pregunta se refiere a **subredes de salud de Bogotá**. Por ello, `No aplica` y `Sin información` no deben tratarse como si fueran una subred territorial comparable.

Esta limpieza responde a la pregunta analítica; no es una eliminación automática de datos.

In [ ]:
# Limpiar SUBRED_SALUD en df

df["SUBRED_SALUD"] = df["SUBRED_SALUD"].str.strip()

In [ ]:
# comprobar

for x in df["SUBRED_SALUD"].dropna().unique():
    print(repr(x))

In [ ]:
# Construir df_bogota

subredes_validas = [
    "Norte",
    "Centro Oriente",
    "Sur",
    "Sur Occidente"
]

df_bogota = df[
    df["SUBRED_SALUD"].isin(subredes_validas)
].copy()

print(df_bogota.shape)
print(df_bogota["SUBRED_SALUD"].value_counts())

In [ ]:
# Reconstruir conteo_subred

conteo_subred = (
    df_bogota
    .groupby(["PERIODO", "SUBRED_SALUD"])
    .size()
    .reset_index(name="EMERGENCIAS")
)

display(conteo_subred.head())

In [ ]:
# Gráfico por facetas

subredes_validas = ["Norte", "Centro Oriente", "Sur", "Sur Occidente"]

df_bogota = df[
    df["SUBRED_SALUD"].isin(subredes_validas)
].copy()

conteo_subred = (
    df_bogota.groupby(["PERIODO", "SUBRED_SALUD"])
             .size()
             .reset_index(name="EMERGENCIAS")
)

g = sns.relplot(
    data=conteo_subred,
    x="PERIODO",
    y="EMERGENCIAS",
    col="SUBRED_SALUD",
    col_wrap=2,
    kind="line",
    marker="o",
    height=3.5,
    aspect=1.25
)

g.set_axis_labels("Periodo", "Número de emergencias")
g.set_titles("Subred: {col_name}")

for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=45)

g.figure.suptitle(
    "Evolución mensual del número de emergencias por subred",
    y=1.03
)
plt.show()

### Pregunta crítica para el aula

> ¿Las facetas aclaran la comparación o habría sido mejor colocar las cuatro líneas en un único gráfico?

**Criterio:** las facetas son útiles cuando separar grupos reduce superposición y permite comparar patrones con la misma estructura. Sin embargo, demasiadas facetas pueden fragmentar la lectura y generar ruido.

Esta es una decisión de diseño, no una regla de Seaborn.

# Parte II. Preparar variables numéricas comparables

## 5. La edad: una variable numérica que no puede usarse todavía

Queremos incorporar el perfil etario al análisis multivariado. Pero `EDAD` no está expresada siempre en la misma unidad.

### Pregunta antes de limpiar

> **¿Qué ocurriría si interpretáramos directamente `EDAD = 6` como seis años cuando la unidad del registro es meses?**

Primero visualizamos `EDAD` sin considerar la unidad.

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df_bogota,
    x="EDAD",
    bins=40,
    kde=True
)

plt.title("Distribución de EDAD — antes de homogeneizar unidades")
plt.xlabel("Valor registrado en EDAD")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
print(df_bogota["UNIDAD"].value_counts(dropna=False))

### Limpieza de edad

Convertiremos todas las edades a **años**. También reconoceremos las variantes dañadas de la palabra “Años” que aparecen en los archivos.

No imputaremos edades faltantes: para los análisis que necesitan edad trabajaremos únicamente con registros donde sea posible obtener una edad comparable.

In [ ]:
def edad_en_anios(fila):
    edad = fila["EDAD"]
    unidad = fila["UNIDAD"]

    if pd.isna(edad) or pd.isna(unidad):
        return np.nan

    unidad = str(unidad).strip().lower()

    # En los archivos existen variantes corruptas de "Años".
    if unidad.startswith("a"):
        return edad
    elif unidad.startswith("mes"):
        return edad / 12
    elif unidad.startswith("dia"):
        return edad / 365.25
    elif unidad.startswith("hora"):
        return edad / (365.25 * 24)

    return np.nan

df_bogota["EDAD_ANIOS"] = df_bogota.apply(edad_en_anios, axis=1)

df_bogota[["EDAD", "UNIDAD", "EDAD_ANIOS"]].dropna().head(15)

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df_bogota,
    x="EDAD_ANIOS",
    bins=40,
    kde=True
)

plt.title("Distribución de edad — unidades homogeneizadas en años")
plt.xlabel("Edad (años)")
plt.ylabel("Frecuencia")
plt.show()

### Interpretación

Compare los dos histogramas. El primero no era necesariamente “incorrecto” desde el punto de vista del código; el problema era **semántico**: mezclaba números que representaban unidades diferentes.

**Principio:** antes de estudiar asociaciones entre variables numéricas, debemos comprobar que las escalas sean comparables.

## 6. Limpiar prioridad para el análisis multivariado

La prioridad también aparece codificada de dos maneras: categorías textuales y valores numéricos.

Primero observamos la variable tal como llega.

In [ ]:
print(df_bogota["PRIORIDAD_FINAL"].value_counts(dropna=False))

Usaremos una codificación común. En estos archivos los códigos numéricos corresponden a:

- 1 → Crítica
- 2 → Alta
- 3 → Media
- 4 → Baja

Crearemos además un indicador de **alta o crítica** para calcular proporciones por localidad y periodo.

In [ ]:
mapa_prioridad = {
    "1": "Critica",
    "2": "Alta",
    "3": "Media",
    "4": "Baja",
    "Critica": "Critica",
    "Alta": "Alta",
    "Media": "Media",
    "Baja": "Baja"
}

df_bogota["PRIORIDAD_LIMPIA"] = (
    df_bogota["PRIORIDAD_FINAL"]
    .astype(str)
    .str.strip()
    .map(mapa_prioridad)
)

df_bogota["ALTA_CRITICA"] = (
    df_bogota["PRIORIDAD_LIMPIA"].isin(["Alta", "Critica"])
).astype(int)

print(df_bogota["PRIORIDAD_LIMPIA"].value_counts(dropna=False))

# Parte III. `pairplot`: muchas relaciones en una sola vista

## 7. Construcción de una tabla analítica

`pairplot` funciona especialmente bien cuando queremos explorar varias variables numéricas simultáneamente. No utilizaremos cada llamada como una observación independiente para esta pregunta; construiremos una tabla **localidad–mes**.

Para cada localidad y periodo calcularemos:

- número de emergencias;
- edad promedio;
- proporción de emergencias de prioridad alta o crítica;
- proporción de registros femeninos con género informado.

### Pregunta analítica 3

> **¿Qué relaciones aparecen entre el volumen mensual de emergencias, la edad promedio y la composición de prioridad y género de las localidades?**

In [ ]:
df_bogota["ES_FEMENINO"] = np.where(
    df_bogota["GENERO"].eq("FEMENINO"), 1,
    np.where(df_bogota["GENERO"].eq("MASCULINO"), 0, np.nan)
)

analisis = (
    df_bogota
    .groupby(["LOCALIDAD", "SUBRED_SALUD", "PERIODO"])
    .agg(
        TOTAL_EMERGENCIAS=("NUMERO_INCIDENTE", "size"),
        EDAD_PROMEDIO=("EDAD_ANIOS", "mean"),
        PROP_ALTA_CRITICA=("ALTA_CRITICA", "mean"),
        PROP_FEMENINO=("ES_FEMENINO", "mean")
    )
    .reset_index()
)

analisis["PCT_ALTA_CRITICA"] = 100 * analisis["PROP_ALTA_CRITICA"]
analisis["PCT_FEMENINO"] = 100 * analisis["PROP_FEMENINO"]

display(analisis.head())
print("Observaciones localidad–mes:", len(analisis))

### Antes de limpiar para `pairplot`

Veamos qué ocurre si enviamos la tabla directamente. Los valores faltantes de edad o género pueden reducir las observaciones disponibles para ciertas relaciones.

In [ ]:
variables = [
    "TOTAL_EMERGENCIAS",
    "EDAD_PROMEDIO",
    "PCT_ALTA_CRITICA",
    "PCT_FEMENINO"
]

print(analisis[variables].isna().sum())

sns.pairplot(
    data=analisis,
    vars=variables,
    corner=True
)

plt.show()

### Limpieza específica

Para comparar exactamente las mismas observaciones en todos los paneles, construiremos una tabla completa para estas cuatro variables.

**Importante:** `dropna()` no se aplica a toda la base. Se aplica únicamente a la tabla y variables requeridas por esta pregunta.

In [ ]:
analisis_completo = analisis.dropna(subset=variables).copy()

print("Antes:", len(analisis))
print("Después:", len(analisis_completo))

In [ ]:
g = sns.pairplot(
    data=analisis_completo,
    vars=variables,
    hue="SUBRED_SALUD",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.65}
)

g.figure.suptitle(
    "Relaciones entre indicadores mensuales por localidad",
    y=1.02
)
plt.show()

### Cómo leer un `pairplot`

- La diagonal muestra la distribución de cada variable.
- Fuera de la diagonal se observan relaciones entre pares de variables.
- El color permite comprobar si los patrones parecen diferir entre subredes.
- No debemos buscar “correlaciones” solo porque haya muchos paneles: debemos volver a la pregunta analítica.

### Pregunta crítica

> ¿El `pairplot` aporta claridad o ya estamos llegando a un punto de sobrecarga visual?

Un `pairplot` es útil para **exploración** y detección de relaciones candidatas. No suele ser la mejor visualización para comunicar un único hallazgo a una audiencia ejecutiva. Esta distinción conecta directamente con los criterios de selección trabajados en la Sesión 10.

# Parte IV. `heatmap`: resumir correlaciones

## 8. De la exploración visual a una matriz de correlaciones

### Pregunta analítica 4

> **¿Cuáles de los indicadores construidos presentan las asociaciones lineales más fuertes?**

Primero calculamos la matriz. Una correlación resume la intensidad y dirección de una **relación lineal** entre dos variables numéricas. No implica causalidad.

In [ ]:
corr = analisis_completo[variables].corr()
corr

### Primera versión: mapa de calor sin decisiones de diseño

Es ejecutable, pero debemos preguntarnos si facilita realmente la lectura.

In [ ]:
sns.heatmap(corr)
plt.title("Matriz de correlaciones — versión inicial")
plt.show()

### Versión depurada para comunicar

Eliminaremos información redundante de la mitad superior, fijaremos una escala comparable entre −1 y 1 y mostraremos los coeficientes.

In [ ]:
mascara = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    mask=mascara,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    cmap="vlag",
    square=True,
    linewidths=.5
)

plt.title("Correlaciones entre indicadores mensuales por localidad")
plt.tight_layout()
plt.show()

### Discusión

Preguntas para el grupo:

1. ¿Qué par de variables presenta la asociación positiva más fuerte?
2. ¿Cuál presenta la asociación negativa más fuerte?
3. ¿Hay asociaciones cercanas a cero?
4. ¿Podemos concluir causalidad? **No.**
5. ¿Qué relación merece ser examinada con un gráfico específico?

**Cuándo aporta claridad:** cuando necesitamos una vista compacta de varias correlaciones.

**Cuándo genera ruido:** cuando hay demasiadas variables, nombres ilegibles o cuando el color sustituye —en lugar de apoyar— la interpretación numérica.

# Parte V. `lmplot`: relaciones lineales

## 9. Del mapa de calor a una relación concreta

Ahora dejamos de mirar todas las combinaciones y seleccionamos una relación para estudiarla con mayor detalle.

### Pregunta analítica 5

> **¿Existe una relación visual entre el volumen mensual de emergencias de una localidad y la proporción de casos clasificados como alta o crítica? ¿Ese patrón parece similar entre subredes?**

`lmplot` combina un diagrama de dispersión con un ajuste de regresión. El objetivo aquí es **visualizar el patrón**, no construir todavía un modelo predictivo.

### Primera versión: todos los puntos en un único panel

Esta versión permite ver la relación general, pero puede ocultar diferencias entre grupos.

In [ ]:
sns.lmplot(
    data=analisis_completo,
    x="TOTAL_EMERGENCIAS",
    y="PCT_ALTA_CRITICA",
    height=5.5,
    aspect=1.3,
    scatter_kws={"alpha": 0.6}
)

plt.xlabel("Número mensual de emergencias")
plt.ylabel("% de emergencias alta o crítica")
plt.title("Volumen y prioridad — relación lineal general")
plt.show()

### Versión con facetas

Ahora combinamos dos ideas centrales de la sesión: **regresión visual + facetas**.

La pregunta ya no es únicamente si existe una relación general, sino si el patrón visual cambia según la subred.

In [ ]:
g = sns.lmplot(
    data=analisis_completo,
    x="TOTAL_EMERGENCIAS",
    y="PCT_ALTA_CRITICA",
    col="SUBRED_SALUD",
    col_wrap=2,
    height=3.6,
    aspect=1.15,
    scatter_kws={"alpha": 0.65}
)

g.set_axis_labels(
    "Número mensual de emergencias",
    "% alta o crítica"
)
g.set_titles("Subred: {col_name}")

g.figure.suptitle(
    "Relación entre volumen y prioridad por subred",
    y=1.03
)
plt.show()

### Pregunta crítica

> ¿La línea de regresión ayuda a revelar un patrón o puede llevarnos a atribuir una relación lineal donde los puntos muestran una estructura diferente?

La línea nunca debe interpretarse aislada de los puntos. Una regresión visual es una ayuda para reconocer estructura, no una prueba causal.

# Parte VI. Relaciones no lineales

## 10. ¿Una recta siempre es suficiente?

La guía de la sesión exige discutir relaciones **lineales y no lineales**. Una manera visual de hacerlo es comparar un ajuste lineal con un ajuste polinómico.

### Pregunta analítica 6

> **¿La relación entre volumen de emergencias y prioridad parece suficientemente representada por una recta o aparece una curvatura que merece exploración?**

Primero mantenemos el ajuste lineal como referencia.

In [ ]:
g = sns.lmplot(
    data=analisis_completo,
    x="TOTAL_EMERGENCIAS",
    y="PCT_ALTA_CRITICA",
    order=1,
    height=5.5,
    aspect=1.25,
    scatter_kws={"alpha": 0.55}
)

plt.xlabel("Número mensual de emergencias")
plt.ylabel("% alta o crítica")
plt.title("Ajuste lineal")
plt.show()

Ahora permitimos una curva polinómica de segundo grado. El hecho de que una curva se adapte visualmente mejor **no significa automáticamente que sea el modelo correcto**.

In [ ]:
g = sns.lmplot(
    data=analisis_completo,
    x="TOTAL_EMERGENCIAS",
    y="PCT_ALTA_CRITICA",
    order=2,
    height=5.5,
    aspect=1.25,
    scatter_kws={"alpha": 0.55}
)

plt.xlabel("Número mensual de emergencias")
plt.ylabel("% alta o crítica")
plt.title("Ajuste no lineal: polinomio de grado 2")
plt.show()

### Comparación guiada

Pregunte al grupo:

- ¿La curvatura revela una estructura evidente o solo sigue variaciones locales?
- ¿Hay suficientes observaciones a lo largo de todo el eje X?
- ¿Los valores extremos están condicionando la forma?
- ¿Cuál de los dos ajustes comunica de manera más prudente lo que realmente muestran los datos?

**Idea clave:** una visualización más compleja no es necesariamente una visualización más informativa.

# Parte VII. Variables numéricas y categóricas

## 11. Comparar la edad según prioridad

Hasta ahora usamos principalmente indicadores numéricos agregados. La guía también solicita visualizaciones comparativas de variables **numéricas y categóricas**.

### Pregunta analítica 7

> **¿El perfil de edad parece diferir entre niveles de prioridad de las emergencias?**

Primero observemos los datos utilizables.

In [ ]:
edad_prioridad = df_bogota[
    ["EDAD_ANIOS", "PRIORIDAD_LIMPIA", "SUBRED_SALUD"]
].dropna().copy()

orden_prioridad = ["Critica", "Alta", "Media", "Baja"]

print(edad_prioridad.shape)
display(edad_prioridad.head())

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=edad_prioridad,
    x="PRIORIDAD_LIMPIA",
    y="EDAD_ANIOS",
    order=orden_prioridad
)

plt.title("Distribución de edad según prioridad")
plt.xlabel("Prioridad")
plt.ylabel("Edad (años)")
plt.show()

Ahora usamos facetas para comprobar si el patrón general se conserva entre subredes.

In [ ]:
g = sns.catplot(
    data=edad_prioridad,
    x="PRIORIDAD_LIMPIA",
    y="EDAD_ANIOS",
    col="SUBRED_SALUD",
    col_wrap=2,
    kind="box",
    order=orden_prioridad,
    height=3.6,
    aspect=1.15
)

g.set_axis_labels("Prioridad", "Edad (años)")
g.set_titles("Subred: {col_name}")

for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=25)

g.figure.suptitle(
    "Edad según prioridad y subred",
    y=1.03
)
plt.show()

### Discusión

Este ejemplo muestra que las facetas no son exclusivas de gráficos de dispersión. Pueden utilizarse para comparar una variable numérica frente a una categórica manteniendo un tercer criterio de agrupación.

Pregunte:

> ¿La faceta añade una comparación relevante para nuestra pregunta o multiplica paneles sin aportar una conclusión nueva?

Esta pregunta debe acompañar cualquier decisión de facetado.

# Parte VIII. Cierre: Seaborn como herramienta analítica, no como catálogo de gráficos

## 12. Volvamos a la pregunta crítica

> **¿Qué patrones territoriales y relaciones entre volumen, perfil de los casos y prioridad de las emergencias pueden identificarse, y cuáles de esos patrones son reales frente a cuáles podrían ser artefactos de problemas de calidad de los datos?**

Para responderla debemos separar dos niveles:

### Nivel 1 — Calidad de datos
Vimos que:
- una localidad podía aparecer fragmentada por problemas de codificación;
- la edad mezclaba años, meses, días y horas;
- la prioridad combinaba etiquetas y códigos;
- existían valores faltantes y categorías territoriales no comparables.

### Nivel 2 — Decisión visual
Después de limpiar únicamente lo necesario:
- **facetas** permitieron comparar patrones entre grupos;
- **`pairplot`** permitió explorar simultáneamente varias relaciones;
- **`heatmap`** resumió correlaciones lineales;
- **`lmplot`** permitió examinar relaciones lineales y explorar posibles formas no lineales;
- los gráficos comparativos permitieron relacionar variables numéricas y categóricas.

### La pregunta final no es “¿qué función de Seaborn sé usar?”

La pregunta es:

> **¿Qué visualización necesito para responder mi pregunta analítica con la menor complejidad necesaria y sin ocultar problemas de los datos?**

## 13. Seaborn y Power BI: herramientas complementarias

El propósito de esta sesión no es reemplazar Power BI.

**Python + Seaborn** resulta especialmente útil para:

- análisis exploratorio reproducible;
- inspección de distribuciones y relaciones;
- comparación rápida de múltiples variables;
- diagnóstico visual antes de decidir qué comunicar.

**Power BI** aporta especialmente:

- interacción con usuarios;
- filtros y navegación;
- publicación y consumo del reporte;
- seguimiento de indicadores dentro de un producto visual.

### Conexión con la Sesión 10

La selección de una visualización sigue dependiendo de:

- la pregunta analítica;
- el tipo de variables;
- la comparación que queremos facilitar;
- la audiencia;
- el mensaje.

La herramienta cambia. **El criterio visual permanece.**

# 14. Actividad de cierre

En parejas, seleccionen **una relación diferente** a la desarrollada durante la clase y respondan:

1. ¿Cuál es su pregunta analítica?
2. ¿Qué variables necesitan?
3. ¿Qué problemas de calidad deben resolver antes de graficar?
4. Construyan una visualización inicial con los datos sin depurar.
5. Realicen únicamente la limpieza necesaria.
6. Construyan la visualización final.
7. Justifiquen por qué eligieron `pairplot`, `heatmap`, `lmplot`, una faceta u otra visualización de Seaborn.
8. Escriban un hallazgo en máximo dos frases.
9. Expliquen una limitación de su análisis.

### Criterio de cierre

No se evalúa quién construye el gráfico más complejo. Se busca que exista coherencia entre:

**pregunta → calidad de datos → selección visual → interpretación → comunicación.**